In [1]:
from datasets import load_dataset

dataset = load_dataset("tomaarsen/conll2003")

print(dataset)

README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

d:\named_entity_recognition_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\datasets--tomaarsen--conll2003. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dataset_infos.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [2]:
# عرض أسماء الـ NER labels
ner_feature = dataset["train"].features["ner_tags"]
label_names = ner_feature.feature.names

print("NER Labels:")
for label_id, label_name in enumerate(label_names):
    print(f"{label_id}: {label_name}")

NER Labels:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC
7: B-MISC
8: I-MISC


In [3]:
# فحص أول جملة في بيانات التدريب
sample = dataset["train"][0]

tokens = sample["tokens"]
ner_ids = sample["ner_tags"]
ner_labels = [label_names[label_id] for label_id in ner_ids]

print("Sentence:")
print(" ".join(tokens))

print("\nTokens:")
print(tokens)

print("\nNER Tag IDs:")
print(ner_ids)

print("\nNER Labels:")
print(ner_labels)

Sentence:
EU rejects German call to boycott British lamb .

Tokens:
['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

NER Tag IDs:
[3, 0, 7, 0, 0, 0, 7, 0, 0]

NER Labels:
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [4]:
print(f"{'Token':20} {'NER Label'}")
print("-" * 35)

for token, label in zip(tokens, ner_labels):
    print(f"{token:20} {label}")

Token                NER Label
-----------------------------------
EU                   B-ORG
rejects              O
German               B-MISC
call                 O
to                   O
boycott              O
British              B-MISC
lamb                 O
.                    O


In [5]:
import numpy as np
import pandas as pd

split_statistics = []

for split_name, split_data in dataset.items():
    sentence_lengths = [len(tokens) for tokens in split_data["tokens"]]

    split_statistics.append({
        "split": split_name,
        "sentences": len(split_data),
        "total_tokens": sum(sentence_lengths),
        "average_length": round(np.mean(sentence_lengths), 2),
        "median_length": round(np.median(sentence_lengths), 2),
        "minimum_length": min(sentence_lengths),
        "maximum_length": max(sentence_lengths),
    })

split_statistics_df = pd.DataFrame(split_statistics)
split_statistics_df

,split,sentences,total_tokens,average_length,median_length,minimum_length,maximum_length
0,train,14041,203621,14.50,10.0,1,113
1,validation,3250,51362,15.80,11.0,1,109
2,test,3453,46435,13.45,9.0,1,124


In [6]:
from collections import Counter

train_tag_counter = Counter()

for tag_ids in dataset["train"]["ner_tags"]:
    train_tag_counter.update(
        label_names[tag_id] for tag_id in tag_ids
    )

total_train_tokens = sum(train_tag_counter.values())

tag_distribution_df = pd.DataFrame([
    {
        "label": label,
        "count": train_tag_counter[label],
        "percentage": round(
            train_tag_counter[label] / total_train_tokens * 100,
            2
        )
    }
    for label in label_names
])

tag_distribution_df

,label,count,percentage
0,O,169578,83.28
1,B-PER,6600,3.24
2,I-PER,4528,2.22
3,B-ORG,6321,3.10
4,I-ORG,3704,1.82
5,B-LOC,7140,3.51
6,I-LOC,1157,0.57
7,B-MISC,3438,1.69
8,I-MISC,1155,0.57


In [7]:
from collections import Counter
import numpy as np
import pandas as pd

# بناء Vocabulary من بيانات التدريب فقط
train_word_counter = Counter(
    token
    for sentence in dataset["train"]["tokens"]
    for token in sentence
)

train_vocab = set(train_word_counter.keys())

print(f"Train vocabulary size: {len(train_vocab):,}")
print(f"Words appearing once: {sum(count == 1 for count in train_word_counter.values()):,}")
print(f"Words appearing twice or less: {sum(count <= 2 for count in train_word_counter.values()):,}")

Train vocabulary size: 23,623
Words appearing once: 11,641
Words appearing twice or less: 15,497


In [8]:
oov_statistics = []

for split_name in ["validation", "test"]:
    split_tokens = [
        token
        for sentence in dataset[split_name]["tokens"]
        for token in sentence
    ]

    oov_tokens = [
        token for token in split_tokens
        if token not in train_vocab
    ]

    oov_statistics.append({
        "split": split_name,
        "total_tokens": len(split_tokens),
        "oov_tokens": len(oov_tokens),
        "unique_oov_words": len(set(oov_tokens)),
        "oov_percentage": round(
            len(oov_tokens) / len(split_tokens) * 100,
            2
        )
    })

oov_statistics_df = pd.DataFrame(oov_statistics)
oov_statistics_df

,split,total_tokens,oov_tokens,unique_oov_words,oov_percentage
0,validation,51362,4293,3260,8.36
1,test,46435,5656,3693,12.18


In [9]:
all_sentence_lengths = [
    len(sentence)
    for split_name in dataset.keys()
    for sentence in dataset[split_name]["tokens"]
]

length_percentiles = pd.DataFrame({
    "percentile": [50, 75, 90, 95, 99, 100],
    "sentence_length": np.percentile(
        all_sentence_lengths,
        [50, 75, 90, 95, 99, 100]
    ).astype(int)
})

length_percentiles

,percentile,sentence_length
0,50,10
1,75,22
2,90,33
3,95,38
4,99,46
5,100,124


In [10]:
from collections import Counter
import pandas as pd


def extract_entities(tag_sequence):
    """
    Extract complete entities from an IOB tag sequence.

    Example:
    B-PER I-PER O B-LOC
    -> PER, LOC
    """
    entities = []
    current_entity_type = None
    entity_start = None

    # Add O to close any entity existing at the end
    for index, tag in enumerate(tag_sequence + ["O"]):

        if tag == "O":
            if current_entity_type is not None:
                entities.append({
                    "entity_type": current_entity_type,
                    "start": entity_start,
                    "end": index - 1
                })

                current_entity_type = None
                entity_start = None

            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B" or entity_type != current_entity_type:
            if current_entity_type is not None:
                entities.append({
                    "entity_type": current_entity_type,
                    "start": entity_start,
                    "end": index - 1
                })

            current_entity_type = entity_type
            entity_start = index

    return entities

In [11]:
entity_statistics = []

for split_name, split_data in dataset.items():
    entity_counter = Counter()

    for tag_ids in split_data["ner_tags"]:
        tags = [label_names[tag_id] for tag_id in tag_ids]
        entities = extract_entities(tags)

        entity_counter.update(
            entity["entity_type"]
            for entity in entities
        )

    total_entities = sum(entity_counter.values())

    for entity_type in ["PER", "ORG", "LOC", "MISC"]:
        entity_statistics.append({
            "split": split_name,
            "entity_type": entity_type,
            "count": entity_counter[entity_type],
            "percentage": round(
                entity_counter[entity_type] / total_entities * 100,
                2
            )
        })

entity_statistics_df = pd.DataFrame(entity_statistics)

entity_statistics_df.pivot(
    index="entity_type",
    columns="split",
    values="count"
)

split,test,train,validation
entity_type,,,
LOC,1668,7140,1837
MISC,702,3438,922
ORG,1661,6321,1341
PER,1617,6600,1842


In [12]:
def find_invalid_iob_transitions(tag_ids):
    invalid_transitions = []
    previous_tag = "O"

    for token_index, tag_id in enumerate(tag_ids):
        current_tag = label_names[tag_id]

        if current_tag.startswith("I-"):
            current_type = current_tag[2:]

            valid_previous_tags = {
                f"B-{current_type}",
                f"I-{current_type}"
            }

            if previous_tag not in valid_previous_tags:
                invalid_transitions.append({
                    "token_index": token_index,
                    "previous_tag": previous_tag,
                    "current_tag": current_tag
                })

        previous_tag = current_tag

    return invalid_transitions


iob_validation_results = []

for split_name, split_data in dataset.items():
    invalid_count = 0

    for tag_ids in split_data["ner_tags"]:
        invalid_count += len(
            find_invalid_iob_transitions(tag_ids)
        )

    iob_validation_results.append({
        "split": split_name,
        "invalid_iob_transitions": invalid_count
    })

pd.DataFrame(iob_validation_results)


,split,invalid_iob_transitions
0,train,0
1,validation,0
2,test,0


In [13]:
import json
from pathlib import Path

# تحديد مسار المشروع سواء الـNotebook تعمل من root أو notebooks
current_path = Path.cwd().resolve()

if (current_path / "reports").exists():
    project_root = current_path
elif (current_path.parent / "reports").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError("Could not locate the project root folder.")

results_dir = project_root / "reports" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# تحويل DataFrames إلى بيانات قابلة للحفظ في JSON
split_stats_records = json.loads(
    split_statistics_df.to_json(orient="records")
)

tag_distribution_records = json.loads(
    tag_distribution_df.to_json(orient="records")
)

oov_stats_records = json.loads(
    oov_statistics_df.to_json(orient="records")
)

length_percentile_records = json.loads(
    length_percentiles.to_json(orient="records")
)

entity_stats_records = json.loads(
    entity_statistics_df.to_json(orient="records")
)

dataset_summary = {
    "dataset_name": "CoNLL-2003",
    "task": "Named Entity Recognition",
    "tagging_scheme": "IOB2",

    "splits": split_stats_records,

    "labels": {
        str(label_id): label_name
        for label_id, label_name in enumerate(label_names)
    },

    "training_vocabulary": {
        "vocabulary_size": len(train_vocab),
        "words_appearing_once": sum(
            count == 1
            for count in train_word_counter.values()
        ),
        "words_appearing_twice_or_less": sum(
            count <= 2
            for count in train_word_counter.values()
        )
    },

    "oov_statistics": oov_stats_records,
    "sentence_length_percentiles": length_percentile_records,
    "train_tag_distribution": tag_distribution_records,
    "entity_distribution": entity_stats_records,

    "invalid_iob_transitions": {
        "train": 0,
        "validation": 0,
        "test": 0
    },

    "recommended_max_sequence_length": 128
}

summary_path = results_dir / "dataset_summary.json"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(
        dataset_summary,
        file,
        indent=4,
        ensure_ascii=False
    )

# حفظ الجداول المهمة أيضًا بصيغة CSV
split_statistics_df.to_csv(
    results_dir / "split_statistics.csv",
    index=False
)

tag_distribution_df.to_csv(
    results_dir / "train_tag_distribution.csv",
    index=False
)

entity_statistics_df.to_csv(
    results_dir / "entity_distribution.csv",
    index=False
)

oov_statistics_df.to_csv(
    results_dir / "oov_statistics.csv",
    index=False
)

print("EDA results saved successfully.")
print(f"Results directory: {results_dir}")
print(f"Summary file: {summary_path}")

EDA results saved successfully.
Results directory: D:\named_entity_recognition_project\reports\results
Summary file: D:\named_entity_recognition_project\reports\results\dataset_summary.json
